# Ensemble: RNN + BiLSTM (ESM-2) with K-Fold Cross-Validation

This notebook trains an ensemble model combining RNN and BiLSTM architectures with pre-computed ESM-2 embeddings using K-fold cross-validation and early stopping.

In [ ]:
import os
import torch, torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from tqdm import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())


## 1. Data + ESM-2

In [ ]:
seq_df = pd.read_csv('data/2018-06-06-pdb-intersect-pisces.csv', usecols=['pdb_id','seq'])
lab_df = pd.read_csv('data/2018-06-06-ss.cleaned.csv', usecols=['pdb_id','sst8','sst3'])
df = pd.merge(seq_df, lab_df, on='pdb_id', how='inner')
df['seq'] = df['seq'].str.replace('*','X')
df = df.dropna(subset=['seq','sst8','sst3']).copy()
df = df[(df['seq'].str.len()==df['sst8'].str.len()) & (df['seq'].str.len()==df['sst3'].str.len())].reset_index(drop=True)
df['len'] = df['seq'].str.len()
print(df.shape)
esm_model, alphabet = torch.hub.load('facebookresearch/esm:main', 'esm2_t30_150M_UR50D')
esm_model.eval().to(device)
batch_converter = alphabet.get_batch_converter()
ss8_vocab = {'H':0,'G':1,'I':2,'E':3,'B':4,'T':5,'S':6,'C':7}
ss3_vocab = {'H':0,'E':1,'C':2}


## 2. Embeddings

In [ ]:
seqs = [s for s in df['seq'].tolist()]
ids = df['pdb_id'].tolist()
pairs = list(zip(ids, seqs))
all_embeddings = []
bs = 8
for i in tqdm(range(0,len(pairs),bs), desc='Generating Embeddings'):
    b = pairs[i:i+bs]
    _,_, toks = batch_converter(b)
    toks = toks.to(device)
    with torch.no_grad():
        out = esm_model(toks, repr_layers=[esm_model.num_layers], return_contacts=False)
    emb = out['representations'][esm_model.num_layers][:,1:-1,:]
    all_embeddings.extend([e.cpu() for e in emb])
padded_embeddings = pad_sequence(all_embeddings, batch_first=True, padding_value=0.0)
embedding_dim = padded_embeddings.shape[-1]
padded_embeddings.shape


## 3. K-Fold Cross-Validation Setup - Labels + Loaders
Prepare data for K-fold cross-validation with train/validation/test splits.

In [ ]:
from sklearn.model_selection import KFold

def enc_labels(col, vocab, L):
    out=[]
    for s in df[col].tolist():
        t=[vocab.get(c,-1) for c in s]
        if len(t)<L: t+=[-1]*(L-len(t))
        else: t=t[:L]
        out.append(torch.tensor(t, dtype=torch.long))
    return pad_sequence(out, batch_first=True, padding_value=-1)

L = padded_embeddings.shape[1]
y8 = enc_labels('sst8', ss8_vocab, L)
y3 = enc_labels('sst3', ss3_vocab, L)

class DS(Dataset):
    def __init__(self, e, a, b): self.e,self.a,self.b=e,a,b
    def __len__(self): return len(self.e)
    def __getitem__(self,i): return self.e[i], self.a[i], self.b[i]

# K-Fold Cross-Validation Setup: Split data into train+val (90%) and test (10%)
n = len(padded_embeddings)
n_test = int(n * 0.1)
test_indices = list(range(n - n_test, n))
train_val_indices = list(range(n - n_test))

# Test dataset
ds_te = DS(padded_embeddings[test_indices], y8[test_indices], y3[test_indices])

# K-Fold configuration
n_folds = 5
kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)

opt = dict(num_workers=2, pin_memory=True)
test_loader = DataLoader(ds_te, batch_size=16, shuffle=False, **opt)

print(f'K-Fold Setup:')
print(f'  Train+Val: {len(train_val_indices)} samples (for K-fold cross-validation)')
print(f'  Test: {len(ds_te)} samples (held-out)')
print(f'  Embedding dimension: {embedding_dim}')


## 4. Models

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_dim, hidden=256, layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, num_layers=layers, bidirectional=False, batch_first=True, dropout=dropout if layers>1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.q8 = nn.Linear(hidden, 8)
        self.q3 = nn.Linear(hidden, 3)
    def forward(self, x, lengths):
        orig = x.size(1)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        po,_ = self.lstm(packed)
        x,_ = pad_packed_sequence(po, batch_first=True, total_length=orig)
        x = self.drop(x)
        return self.q8(x), self.q3(x)

class BiLSTM(nn.Module):
    def __init__(self, input_dim, hidden=256, dropout=0.3):
        super().__init__()
        self.l1 = nn.LSTM(input_dim, hidden, bidirectional=True, batch_first=True)
        self.l2 = nn.LSTM(hidden*2, hidden, bidirectional=True, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.q8 = nn.Linear(hidden*2, 8)
        self.q3 = nn.Linear(hidden*2, 3)
    def forward(self, x, lengths):
        orig = x.size(1)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        po,_ = self.l1(packed)
        po,_ = self.l2(po)
        x,_ = pad_packed_sequence(po, batch_first=True, total_length=orig)
        x = self.drop(x)
        return self.q8(x), self.q3(x)

rnn = RNN(embedding_dim).to(device)
bilstm = BiLSTM(embedding_dim).to(device)
if torch.cuda.device_count()>1: rnn = nn.DataParallel(rnn); bilstm = nn.DataParallel(bilstm)
rnn, bilstm


## 5. Training with K-Fold Cross-Validation and Early Stopping
**Training Configuration:**
- **K-Fold Cross-Validation**: 5 folds
- **Epochs**: 50 epochs per fold
- **Early Stopping**: Patience of 5 epochs (stops if validation accuracy doesn't improve)
- **Models**: RNN and BiLSTM trained separately
- **Embeddings**: Pre-computed ESM-2 embeddings (frozen)
- **Ensemble**: Weighted combination of predictions tuned on validation set

In [ ]:
from torch.utils.data import Subset

def acc(logits, y):
    p = logits.argmax(-1)
    m = y>=0
    return (p[m]==y[m]).float().mean().item() if m.any() else 0.0

def train_model_kfold(model_class, model_name):
    """Train a model using K-fold cross-validation"""
    
    fold_results = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(train_val_indices)):
        print(f"\n{'='*60}")
        print(f"{model_name} - Fold {fold_idx + 1}/{n_folds}")
        print(f"{'='*60}")
        
        # Create datasets for this fold
        train_fold_indices = [train_val_indices[i] for i in train_idx]
        val_fold_indices = [train_val_indices[i] for i in val_idx]
        
        ds_tr = DS(padded_embeddings[train_fold_indices], y8[train_fold_indices], y3[train_fold_indices])
        ds_va = DS(padded_embeddings[val_fold_indices], y8[val_fold_indices], y3[val_fold_indices])
        
        train_loader = DataLoader(ds_tr, batch_size=16, shuffle=True, **opt)
        val_loader = DataLoader(ds_va, batch_size=16, shuffle=False, **opt)
        
        print(f"Fold {fold_idx + 1} - Train: {len(ds_tr)}, Val: {len(ds_va)}")
        
        # Initialize model for this fold
        model = model_class().to(device)
        
        # Training setup
        c8 = nn.CrossEntropyLoss(ignore_index=-1)
        c3 = nn.CrossEntropyLoss(ignore_index=-1)
        opt_adam = torch.optim.Adam(model.parameters(), lr=1e-4)
        
        best_val_acc = 0.0
        epochs_no_improve = 0
        patience = 5
        epochs = 50
        
        for ep in range(1, epochs + 1):
            # Training
            model.train()
            tl = ta8 = ta3 = 0.0
            for e, s8, s3 in tqdm(train_loader, desc=f'{model_name} Fold {fold_idx+1} Epoch {ep}/{epochs}', leave=False):
                e, s8, s3 = e.to(device), s8.to(device), s3.to(device)
                L = (s8>=0).sum(1).to(torch.int64)
                q8, q3 = model(e, L)
                l8 = c8(q8.view(-1, 8), s8.view(-1))
                l3 = c3(q3.view(-1, 3), s3.view(-1))
                loss = l8 + 0.5 * l3
                opt_adam.zero_grad()
                loss.backward()
                opt_adam.step()
                tl += loss.item()
                ta8 += acc(q8, s8)
                ta3 += acc(q3, s3)
            
            tl /= len(train_loader)
            ta8 /= len(train_loader)
            ta3 /= len(train_loader)
            
            # Validation
            model.eval()
            vl = va8 = va3 = 0.0
            with torch.no_grad():
                for e, s8, s3 in val_loader:
                    e, s8, s3 = e.to(device), s8.to(device), s3.to(device)
                    L = (s8>=0).sum(1).to(torch.int64)
                    q8, q3 = model(e, L)
                    l8 = c8(q8.view(-1, 8), s8.view(-1))
                    l3 = c3(q3.view(-1, 3), s3.view(-1))
                    loss = l8 + 0.5 * l3
                    vl += loss.item()
                    va8 += acc(q8, s8)
                    va3 += acc(q3, s3)
            
            vl /= len(val_loader)
            va8 /= len(val_loader)
            va3 /= len(val_loader)
            
            print(f'Epoch {ep}: Train Loss={tl:.4f}, Val Loss={vl:.4f}')
            print(f'Train Acc Q8={ta8:.4f}, Val Acc Q8={va8:.4f}')
            print(f'Train Acc Q3={ta3:.4f}, Val Acc Q3={va3:.4f}')
            
            # Early stopping check
            if va8 > best_val_acc:
                best_val_acc = va8
                epochs_no_improve = 0
                ckpt = f'best_{model_name.lower()}_esm2_fold{fold_idx+1}.pt'
                torch.save(model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(), ckpt)
                print(f'  ✓ Saved best checkpoint for fold {fold_idx+1}')
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(f'  Early stopping triggered at epoch {ep}')
                    break
        
        fold_results.append({
            'fold': fold_idx + 1,
            'best_val_acc': best_val_acc
        })
        
        print(f"\nFold {fold_idx + 1} Best Val Acc Q8: {best_val_acc:.4f}")
    
    # Summary
    print(f"\n{'='*60}")
    print(f"{model_name} K-Fold Cross-Validation Summary")
    print(f"{'='*60}")
    for result in fold_results:
        print(f"Fold {result['fold']}: Val Acc Q8 = {result['best_val_acc']:.4f}")
    
    avg_val_acc = sum([r['best_val_acc'] for r in fold_results]) / n_folds
    print(f"\nAverage Validation Accuracy Q8: {avg_val_acc:.4f}")
    
    # Select best fold
    best_fold = max(fold_results, key=lambda x: x['best_val_acc'])
    print(f"Best Fold: {best_fold['fold']} with Val Acc Q8: {best_fold['best_val_acc']:.4f}")
    
    return best_fold['fold'], fold_results

# Train RNN with K-fold
print("\n" + "="*60)
print("Training RNN with K-Fold Cross-Validation")
print("="*60)
best_rnn_fold, rnn_results = train_model_kfold(ProteinRNN, 'RNN')

# Train BiLSTM with K-fold
print("\n" + "="*60)
print("Training BiLSTM with K-Fold Cross-Validation")
print("="*60)
best_bilstm_fold, bilstm_results = train_model_kfold(ProteinBiLSTM, 'BiLSTM')

# Load best models from best folds
rnn = ProteinRNN().to(device)
rnn.load_state_dict(torch.load(f'best_rnn_esm2_fold{best_rnn_fold}.pt', map_location=device))
rnn.eval()

bilstm = ProteinBiLSTM().to(device)
bilstm.load_state_dict(torch.load(f'best_bilstm_esm2_fold{best_bilstm_fold}.pt', map_location=device))
bilstm.eval()

print(f"\nLoaded best RNN from fold {best_rnn_fold}")
print(f"Loaded best BiLSTM from fold {best_bilstm_fold}")

'trained'


## 6. Ensemble Evaluation
Combines predictions from the best RNN and BiLSTM models (from K-fold cross-validation) using optimized alpha weights tuned on validation set, and evaluates on test set.

In [ ]:
@torch.no_grad()
def collect(loader):
    q8_r,q3_r,q8_b,q3_b,s8s,s3s=[],[],[],[],[],[]
    for e,s8,s3 in loader:
        e = e.to(device); s8s.append(s8); s3s.append(s3)
        L = (s8>=0).sum(1).to(torch.int64)
        a8,a3 = rnn(e,L); b8,b3 = bilstm(e,L)
        q8_r.append(a8.cpu()); q3_r.append(a3.cpu()); q8_b.append(b8.cpu()); q3_b.append(b3.cpu())
    return (torch.cat(q8_r), torch.cat(q3_r), torch.cat(q8_b), torch.cat(q3_b), torch.cat(s8s), torch.cat(s3s))

def masked_acc(logits, labels):
    p = logits.argmax(-1); m = labels>=0
    return (p[m]==labels[m]).float().mean().item() if m.any() else 0.0

v_q8r, v_q3r, v_q8b, v_q3b, v_s8, v_s3 = collect(val_loader)
alphas = torch.linspace(0,1,21)
best_a8=0.5; best_v8=-1; best_a3=0.5; best_v3=-1
for a in alphas:
    acc8 = masked_acc(a*v_q8r + (1-a)*v_q8b, v_s8)
    if acc8>best_v8: best_v8, best_a8 = acc8, float(a)
    acc3 = masked_acc(a*v_q3r + (1-a)*v_q3b, v_s3)
    if acc3>best_v3: best_v3, best_a3 = acc3, float(a)
print(f'Best alpha Q8: {best_a8:.2f} | Val Acc: {best_v8:.4f}')
print(f'Best alpha Q3: {best_a3:.2f} | Val Acc: {best_v3:.4f}')
@torch.no_grad()
def evaluate(loader, a8, a3):
    tot8=tot3=0; cor8=cor3=0
    for e,s8,s3 in loader:
        e,s8,s3 = e.to(device), s8, s3
        L = (s8>=0).sum(1).to(torch.int64)
        r8,r3 = rnn(e,L); b8,b3 = bilstm(e,L)
        q8 = a8*r8 + (1-a8)*b8; q3 = a3*r3 + (1-a3)*b3
        p8 = q8.argmax(-1); p3 = q3.argmax(-1)
        m8 = s8>=0; m3 = s3>=0
        cor8 += (p8[m8]==s8[m8]).sum().item(); tot8 += m8.sum().item()
        cor3 += (p3[m3]==s3[m3]).sum().item(); tot3 += m3.sum().item()
    return cor8/max(1,tot8), cor3/max(1,tot3)
t8,t3 = evaluate(test_loader, best_a8, best_a3)
print(f'Test Accuracy Q8: {t8:.4f}')
print(f'Test Accuracy Q3: {t3:.4f}')
